In [10]:
import os
import gc
import time
import pandas as pd
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

# ==============================================================================
# 🎛️ PARAMETER KONFIGURASI JALUR LOKAL & FILTER 
# ==============================================================================
#PATH_KATALOG_MASTER = '/Volumes/Local Disk/Code_Git/S3_code/seismic/usgs_katalog/katalog_usgs_master_2001_2025.csv'
#PATH_KATALOG_MASTER = '/Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/BMKG_Earthquake_Catalog.csv'
#PATH_KATALOG_MASTER = '/Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/HYBRID_EARTHQUAKE_CATALOG_2001_2024_FIX.csv'
#PATH_KATALOG_MASTER =  '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/input/query_usgs_indonesia_katalog_raw.csv'
PATH_KATALOG_MASTER = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/usgs_events_indonesia_polygon_buffer2deg.csv'

PATH_CSV_FINAL = '/Volumes/Local Disk/Code_Git/S3_code/seismic/waveform_indonesia_usgs_bmkg_katalog/katalog_radar_dll/INDONESIA_STATION_INVENTORY_FINAL.csv'
OUTPUT_WAVEFORM_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/waveform_buffer/waveform_ready'

LOG_ERROR_MURNI_PATH = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/log_data/error_log_murni_0109.csv'
LOG_SUCCESS_PATH = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/log_data/success_log_0109.csv'
LOG_FALLBACK_GFZ_PATH = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/log_data/failed_iris_need_gfz_0109.csv'

# ==============================================================================
# 🎚️ PARAMETER AMBANG BATAS TARGET BARU TAHUN 2006-2007 & KUOTA 200 DATA
# ==============================================================================
START_YEAR = 2018
END_YEAR = 2019
MIN_MAGNITUDE = 3.5
MAX_MAGNITUDE = 9.5

# 📌 TARGET KUOTA BARU: Maksimal mengambil 200 baris rekaman komponen per Event ID
MAX_DATA_PER_EVENT = 200 
MAX_WORKERS = 10

ACADEMIC_USER_AGENT = (
    "ResearchProject: Doctoral Dissertation in AI and Edge Computing; "
    "Researcher: Very Kurnia Bakti (Indonesia); "
    "ID: Scopus:57209452703"
)

clients = {
    "IRIS": Client("IRIS", timeout=60, user_agent=ACADEMIC_USER_AGENT),
    "GFZ": Client("IRIS", timeout=60, user_agent=ACADEMIC_USER_AGENT)
}

COMPLETED_TASKS_SET = set()

def download_single_waveform(task_data):
    eid, time_str, eq_lat, eq_lon, net, sta, server = task_data
    task_key = f"{eid}_{net}_{sta}"
    
    if task_key in COMPLETED_TASKS_SET:
        return "SKIPPED", None
        
    try:
        t_event = UTCDateTime(time_str)
        year_folder = str(t_event.year)
        event_dir = os.path.join(OUTPUT_WAVEFORM_DIR, year_folder, str(eid))
        filename = f"{net}_{sta}_{eid}.mseed"
        file_path = os.path.join(event_dir, filename)
        
        if os.path.exists(file_path) and os.path.getsize(file_path) > 0:
            return "SKIPPED", None
            
        cl = clients.get(server)
        if not cl:
            return "ERROR_SERVER", (eid, time_str, net, sta, server, "Server Config Error")
            
        start_time = t_event - 60
        end_time = t_event + 240
        
        # Ambil data gelombang mentah dan tampung sementara di RAM
        st = cl.get_waveforms(network=net, station=sta, location="*", channel="BH*,HH*,EH*",
                              starttime=start_time, endtime=end_time)
        
        # ==============================================================================
        # 🛡️ SENSOR PENGAMAN: VALIDASI KELENGKAPAN TRI-KOMPONEN (N, Z, E / 1, 2, Z)
        # ==============================================================================
        # Ambil huruf terakhir dari nama channel (e.g., 'BHZ' -> 'Z', 'BHN' -> 'N')
        found_channels = set([tr.stats.channel[-1].upper() for tr in st])
        
        # Definisikan subset komponen lengkap standar seismologi global
        has_standard_nze = {'Z', 'N', 'E'}.issubset(found_channels)
        has_orthogonal_z12 = {'Z', '1', '2'}.issubset(found_channels)
        
        # Jika fasa komponen tidak lengkap, gugurkan penulisan file fisik ke SSD Mac
        if not (has_standard_nze or has_orthogonal_z12):
            channels_logged = ",".join(list(found_channels))
            # Hancurkan stream dari RAM agar terhindar dari memory leak
            del st
            return "FAILED_INCOMPLETE_CHANNELS", (eid, sta, f"Missing Components (Found: {channels_logged})")
        # ==============================================================================
        
        # Jika lolos sensor kelengkapan, eksekusi pembuatan folder dan simpan biner .mseed
        os.makedirs(event_dir, exist_ok=True)
        st.write(file_path, format="MSEED")
        del st
        
        time.sleep(0.05) 
        return "SUCCESS", task_key
            
    except Exception as e:
        err_name = type(e).__name__
        if err_name in ["HTTPError", "FDSNTimeoutException", "ConnectionError", "TimeoutError"]:
            return "NEED_GFZ", (eid, time_str, net, sta, server, err_name)
        else:
            return f"FAILED_{err_name}", (eid, time_str, net, sta, server, err_name)

def build_success_index_from_storage():
    global COMPLETED_TASKS_SET
    print("🔍 Menginisialisasi Indeks Turbo Resume...")
    os.makedirs(os.path.dirname(LOG_SUCCESS_PATH), exist_ok=True)
    
    if os.path.exists(LOG_SUCCESS_PATH):
        try:
            df_succ = pd.read_csv(LOG_SUCCESS_PATH)
            COMPLETED_TASKS_SET = set(df_succ['Task_Key'].astype(str).tolist())
            print(f"✅ Berhasil memuat {len(COMPLETED_TASKS_SET):,} file sukses ke RAM Mac!")
            return
        except Exception:
            pass

    print("📂 Menyisir folder Local Disk untuk mendata file sukses...")
    scanned_keys = []
    if os.path.exists(OUTPUT_WAVEFORM_DIR):
        for root, _, files in os.walk(OUTPUT_WAVEFORM_DIR):
            for file in files:
                if file.endswith('.mseed'):
                    parts = file.replace('.mseed', '').split('_')
                    if len(parts) >= 3:
                        scanned_keys.append(f"{parts[2]}_{parts[0]}_{parts[1]}")
                        
    COMPLETED_TASKS_SET = set(scanned_keys)
    if scanned_keys:
        pd.DataFrame({"Task_Key": scanned_keys}).to_csv(LOG_SUCCESS_PATH, index=False)
    print(f"✅ Sinkronisasi Selesai! {len(COMPLETED_TASKS_SET):,} file terdata di RAM.")

def run_closest_station_pipeline():
    print(f"🛡️  Starting Fixed-Count Seismology Downloader Pipeline (Max {MAX_DATA_PER_EVENT} Data Per Event)...")
    os.makedirs(os.path.dirname(LOG_ERROR_MURNI_PATH), exist_ok=True)
    os.makedirs(os.path.dirname(LOG_FALLBACK_GFZ_PATH), exist_ok=True)
    
    build_success_index_from_storage()
    
    if not os.path.exists(PATH_CSV_FINAL) or not os.path.exists(PATH_KATALOG_MASTER):
        print("❌ Berkas peta navigasi inventory final atau katalog master tidak ditemukan!")
        return
        
    print("⏳ Loading master navigation footprint & earthquake catalog...")
    df_inventory = pd.read_csv(PATH_CSV_FINAL)
    df_master = pd.read_csv(PATH_KATALOG_MASTER)
    
    col_master_id = next((c for c in df_master.columns if 'id' in c.lower()), 'id')
    col_master_lat = next((c for c in df_master.columns if 'lat' in c.lower()), 'latitude')
    col_master_lon = next((c for c in df_master.columns if 'lon' in c.lower()), 'longitude')
    
    df_master_clean = df_master[[col_master_id, col_master_lat, col_master_lon]].copy()
    df_master_clean.columns = ['Event_ID', 'Eq_Latitude', 'Eq_Longitude']
    
    df_inventory['Event_ID'] = df_inventory['Event_ID'].astype(str)
    df_master_clean['Event_ID'] = df_master_clean['Event_ID'].astype(str)
    
    df_merged = pd.merge(df_inventory, df_master_clean, on='Event_ID', how='inner')
    
    if len(df_merged) == 0:
        print("❌ Gagal mencocokkan data! Tidak ada Event_ID yang selaras antara kedua file.")
        return

    col_time = 'Time_UTC'
    col_mag = 'Mag'
    
    df_merged[col_time] = pd.to_datetime(df_merged[col_time], errors='coerce')
    df_filtered = df_merged[
        (df_merged[col_time] >= f"{START_YEAR}-01-01") & 
        (df_merged[col_time] <= f"{END_YEAR}-12-31 23:59:59") &
        (df_merged[col_mag] >= MIN_MAGNITUDE) & 
        (df_merged[col_mag] <= MAX_MAGNITUDE)
    ].copy()
    
    if len(df_filtered) == 0:
        print("⚠️ Tidak ada data yang cocok dengan kriteria rentang target di memori.")
        return

    # Penomoran baris rekaman kumulatif alami per Event_ID (Stasiun dibebaskan)
    df_filtered['Data_Rank'] = df_filtered.groupby('Event_ID').cumcount() + 1
    
    # Saring kuota tugas secara konsisten maksimal 200 data per event gempa
    df_final_tasks = df_filtered[df_filtered['Data_Rank'] <= MAX_DATA_PER_EVENT].copy()
    
    event_counts = df_final_tasks.groupby('Event_ID').size()
    valid_eids = event_counts[event_counts >= 2].index
    df_final_tasks = df_final_tasks[df_final_tasks['Event_ID'].isin(valid_eids)]
    
    tasks = list(df_final_tasks[['Event_ID', 'Time_UTC', 'Eq_Latitude', 'Eq_Longitude', 'Net', 'Station', 'Server']].itertuples(index=False, name=None))
    total_tasks = len(tasks)
    unique_events = len(valid_eids)
    
    del df_master, df_inventory, df_master_clean, df_merged, df_filtered, df_final_tasks
    gc.collect()
    
    print(f"📊 Filter Mengunci: {unique_events:,} Kejadian Gempa Bumi Unik.")
    print(f"📡 Total Antrean Unduhan Sinyal (Maksimal {MAX_DATA_PER_EVENT} Data per Event): {total_tasks:,} Berkas Tugas.")
    
    stats = {"SUCCESS": 0, "SKIPPED": 0, "FAILED": 0, "FALLBACK": 0}
    error_logs, fallback_logs, new_success_keys = [], [], []
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        pbar = tqdm(total=total_tasks, unit="file", desc="Harvesting Waveforms")
        
        for status, detail in executor.map(download_single_waveform, tasks):
            if status == "SUCCESS":
                stats["SUCCESS"] += 1
                if detail: new_success_keys.append({"Task_Key": detail})
            elif status == "SKIPPED":
                stats["SKIPPED"] += 1
            elif status == "NEED_GFZ":
                stats["FALLBACK"] += 1
                if detail: fallback_logs.append({"Event_ID": detail[0], "Time_UTC": detail[1], "Net": detail[2], "Station": detail[3], "Server": "GFZ", "Reason": detail[5]})
            else:
                stats["FAILED"] += 1
                if detail: error_logs.append({"Event_ID": detail[0], "Station": detail[1], "Error_Reason": detail[2]})
                
            pbar.update(1)
            pbar.set_postfix({"New": stats["SUCCESS"], "Skip": stats["SKIPPED"], "To_GFZ": stats["FALLBACK"], "NoData": stats["FAILED"]})
            
            total_processed = stats["SUCCESS"] + stats["SKIPPED"] + stats["FAILED"] + stats["FALLBACK"]
            if total_processed % 400 == 0:
                gc.collect()
                if new_success_keys and len(new_success_keys) >= 1000:
                    pd.DataFrame(new_success_keys).to_csv(LOG_SUCCESS_PATH, mode='a', header=not os.path.exists(LOG_SUCCESS_PATH), index=False)
                    new_success_keys.clear()
                if fallback_logs and len(fallback_logs) >= 500:
                    pd.DataFrame(fallback_logs).to_csv(LOG_FALLBACK_GFZ_PATH, mode='a', header=not os.path.exists(LOG_FALLBACK_GFZ_PATH), index=False)
                    fallback_logs.clear()

    pbar.close()
    
    if new_success_keys: pd.DataFrame(new_success_keys).to_csv(LOG_SUCCESS_PATH, mode='a', header=not os.path.exists(LOG_SUCCESS_PATH), index=False)
    if fallback_logs: pd.DataFrame(fallback_logs).to_csv(LOG_FALLBACK_GFZ_PATH, mode='a', header=not os.path.exists(LOG_FALLBACK_GFZ_PATH), index=False)
    if error_logs: pd.DataFrame(error_logs).to_csv(LOG_ERROR_MURNI_PATH, index=False)
        
    print("\n" + "="*50 + "\n🏁 PIPELINE PENGUNDUHAN KOREKSI KELOMPOK SELESAI\n" + "="*50)
    print(f"✅ Berkas Baru Sukses Terjemput      : {stats['SUCCESS']:,} file")
    print(f"🔄 Berkas Lama Aman Terlewati        : {stats['SKIPPED']:,} file")
    print(f"⚠️ Masalah Jaringan (Dialihkan GFZ) : {stats['FALLBACK']:,} file")
    print(f"❌ Berkas Gagal (Absen/Tidak Lengkap): {stats['FAILED']:,} file")
    print("="*50)

if __name__ == "__main__":
    run_closest_station_pipeline()

/opt/homebrew/Caskroom/miniforge/base/envs/mcu_quake_env/lib/python3.10/site-packages/obspy/clients/fdsn/client.py:251: ObsPyDeprecationWarning: IRIS is now EarthScope, please consider changing the FDSN client short URL to 'EARTHSCOPE'.
  warnings.warn(msg, ObsPyDeprecationWarning)


🛡️  Starting Fixed-Count Seismology Downloader Pipeline (Max 200 Data Per Event)...
🔍 Menginisialisasi Indeks Turbo Resume...
✅ Berhasil memuat 16,492 file sukses ke RAM Mac!
⏳ Loading master navigation footprint & earthquake catalog...
📊 Filter Mengunci: 13 Kejadian Gempa Bumi Unik.
📡 Total Antrean Unduhan Sinyal (Maksimal 200 Data per Event): 156 Berkas Tugas.


Harvesting Waveforms: 100%|██████████| 156/156 [00:09<00:00, 17.18file/s, New=0, Skip=26, To_GFZ=0, NoData=130]


🏁 PIPELINE PENGUNDUHAN KOREKSI KELOMPOK SELESAI
✅ Berkas Baru Sukses Terjemput      : 0 file
🔄 Berkas Lama Aman Terlewati        : 26 file
⚠️ Masalah Jaringan (Dialihkan GFZ) : 0 file
❌ Berkas Gagal (Absen/Tidak Lengkap): 130 file


In [2]:
import os
import gc
import time
import pandas as pd
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# ==============================================================================
# 🎛️ KONFIGURASI JALUR & PARAMETER DINAMIS
# ==============================================================================
PATH_KATALOG_MASTER = '/Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/HYBRID_EARTHQUAKE_CATALOG_2001_2024_FIX.csv'
OUTPUT_WAVEFORM_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/waveform_buffer/waveform_ready'

TARGET_YEAR = 2019
MIN_MAGNITUDE = 3.5   
SEARCH_RADIUS_DEG = 3.0
MAX_STATIONS_PER_EVENT = 100 
MAX_WORKERS = 20 

LOG_DYNAMIC_SUCCESS = f'/Volumes/Local Disk/Code_Git/S3_code/seismic/waveform_buffer/log_success_dynamic_{TARGET_YEAR}.csv'

ACADEMIC_USER_AGENT = "ResearchProject: Doctoral Dissertation in Edge AI; Researcher: Very Kurnia Bakti"
client = Client("EARTHSCOPE", timeout=45, user_agent=ACADEMIC_USER_AGENT)

# ==============================================================================
# 🛠️ FUNGSI PENGUNDUH DINAMIS BERBASIS RADIUS
# ==============================================================================
def download_dynamic_event(task_data):
    eid, time_str, eq_lat, eq_lon, mag = task_data
    sukses_unduh = 0
    stasiun_gagal = 0
    
    try:
        t_event = UTCDateTime(time_str)
        year_folder = str(t_event.year)
        event_dir = os.path.join(OUTPUT_WAVEFORM_DIR, year_folder, str(eid))
        
        # 1. DISCOVERY: Tanya EARTHSCOPE stasiun aktif
        try:
            inventory = client.get_stations(
                starttime=t_event - 60, 
                endtime=t_event + 240, 
                latitude=eq_lat, 
                longitude=eq_lon, 
                maxradius=SEARCH_RADIUS_DEG,
                channel="BH*,HH*,EH*",
                level="station"
            )
        except Exception:
            return eid, 0, 0, "Gagal/Tidak ada stasiun di radius tersebut"

        available_stations = [(network.code, station.code) for network in inventory for station in network]
        
        # 2. HARVESTING
        for net, sta in available_stations:
            if sukses_unduh >= MAX_STATIONS_PER_EVENT:
                break 
                
            filename = f"{net}_{sta}_{eid}.mseed"
            file_path = os.path.join(event_dir, filename)
            
            if os.path.exists(file_path) and os.path.getsize(file_path) > 0:
                sukses_unduh += 1
                continue
                
            try:
                st = client.get_waveforms(
                    network=net, station=sta, location="*", channel="BH*,HH*,EH*",
                    starttime=t_event - 60, endtime=t_event + 240
                )
                
                # SENSOR QC
                found_channels = set([tr.stats.channel[-1].upper() for tr in st])
                has_standard_nze = {'Z', 'N', 'E'}.issubset(found_channels)
                has_orthogonal_z12 = {'Z', '1', '2'}.issubset(found_channels)
                
                if not (has_standard_nze or has_orthogonal_z12):
                    del st
                    stasiun_gagal += 1
                    continue
                    
                os.makedirs(event_dir, exist_ok=True)
                st.write(file_path, format="MSEED")
                del st
                sukses_unduh += 1
                time.sleep(0.1) 
                
            except Exception:
                stasiun_gagal += 1
                continue
                
        return eid, sukses_unduh, stasiun_gagal, "Selesai"
        
    except Exception as e:
        return eid, 0, 0, f"Error: {str(e)}"

# ==============================================================================
# 🚀 PIPELINE UTAMA
# ==============================================================================
def run_dynamic_pipeline():
    print(f"📡 Memulai Dynamic Radius Harvesting untuk Tahun {TARGET_YEAR}...")
    df_master = pd.read_csv(PATH_KATALOG_MASTER)
    
    # MENGGUNAKAN NAMA KOLOM EKSAK
    col_id = 'event_id'
    col_lat = 'latitude'
    col_lon = 'longitude'
    col_time = 'time_utc'
    col_mag = 'magnitude'
    
    df_master['waktu_dt'] = pd.to_datetime(df_master[col_time], errors='coerce', format='mixed')
    
    df_target = df_master[
        (df_master['waktu_dt'].dt.year == TARGET_YEAR) & 
        (df_master[col_mag] >= MIN_MAGNITUDE)
    ].copy()
    
    if len(df_target) == 0:
        print(f"❌ Tidak ada data kejadian gempa untuk tahun {TARGET_YEAR} setelah filter magnitudo.")
        return
        
    tasks = list(df_target[[col_id, col_time, col_lat, col_lon, col_mag]].itertuples(index=False, name=None))
    print(f"📊 Ditemukan {len(tasks)} kejadian gempa target (Mag >= {MIN_MAGNITUDE}).")
    
    total_sukses = 0
    log_results = []
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(download_dynamic_event, task): task for task in tasks}
        
        with tqdm(total=len(tasks), desc="Processing Events", unit="event") as pbar:
            for future in as_completed(futures):
                eid, sukses, gagal, status = future.result()
                total_sukses += sukses
                log_results.append({'Event_ID': eid, 'File_Sukses': sukses, 'File_Gagal': gagal, 'Status': status})
                
                pbar.set_postfix({"Sukses Unduh": total_sukses})
                pbar.update(1)
                
                if len(log_results) % 50 == 0:
                    gc.collect()

    pd.DataFrame(log_results).to_csv(LOG_DYNAMIC_SUCCESS, index=False)
    print("\n" + "="*50)
    print(f"🏁 DYNAMIC HARVESTING {TARGET_YEAR} SELESAI")
    print(f"✅ Total gelombang 3-komponen berhasil diunduh: {total_sukses} file")
    print("="*50)

if __name__ == "__main__":
    run_dynamic_pipeline()

📡 Memulai Dynamic Radius Harvesting untuk Tahun 2019...
📊 Ditemukan 6325 kejadian gempa target (Mag >= 3.5).


Processing Events: 100%|██████████| 6325/6325 [12:12<00:00,  8.64event/s, Sukses Unduh=1380]


🏁 DYNAMIC HARVESTING 2019 SELESAI
✅ Total gelombang 3-komponen berhasil diunduh: 1380 file


In [1]:
import os
import gc
import time
import pandas as pd
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# ==============================================================================
# 🎛️ KONFIGURASI JALUR & PARAMETER DINAMIS (RENTANG TAHUN)
# ==============================================================================
PATH_KATALOG_MASTER = '/Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/HYBRID_EARTHQUAKE_CATALOG_2001_2024_FIX.csv'
OUTPUT_WAVEFORM_DIR = '/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/waveform_buffer/waveform_ready_02'

# --- PARAMETER RENTANG WAKTU & FILTER ---
START_YEAR = 2006
END_YEAR = 2024
MIN_MAGNITUDE = 2.5   
SEARCH_RADIUS_DEG = 5.0       # Radius ~333 km (Sangat ideal untuk M 3.5)
MAX_STATIONS_PER_EVENT = 100  # Kuota stasiun per kejadian gempa
MAX_WORKERS = 15               # Batas aman agar tidak diblokir server EarthScope
TARGET_NETWORKS = "IA,GE,II"  # Filter khusus stasiun yang relevan di Indonesia

LOG_DYNAMIC_SUCCESS = f'/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/log_success_dynamic_02{START_YEAR}_{END_YEAR}.csv'

ACADEMIC_USER_AGENT = "ResearchProject: Doctoral Dissertation in Edge AI; Researcher: Very Kurnia Bakti"
client = Client("EARTHSCOPE", timeout=60, user_agent=ACADEMIC_USER_AGENT)

# ==============================================================================
# 🛠️ FUNGSI PENGUNDUH DINAMIS BERBASIS RADIUS & NETWORK
# ==============================================================================
def download_dynamic_event(task_data):
    eid, time_str, eq_lat, eq_lon, mag = task_data
    sukses_unduh = 0
    stasiun_gagal = 0
    
    try:
        t_event = UTCDateTime(time_str)
        year_folder = str(t_event.year)
        event_dir = os.path.join(OUTPUT_WAVEFORM_DIR, year_folder, str(eid))
        
        # 1. DISCOVERY: Cari stasiun aktif dalam radius & jaringan spesifik
        try:
            inventory = client.get_stations(
                network=TARGET_NETWORKS,
                starttime=t_event - 60, 
                endtime=t_event + 240, 
                latitude=eq_lat, 
                longitude=eq_lon, 
                maxradius=SEARCH_RADIUS_DEG,
                channel="BH*,HH*,EH*",
                level="station"
            )
        except Exception:
            return eid, 0, 0, "Gagal/Tidak ada stasiun di radius tersebut"

        available_stations = [(network.code, station.code) for network in inventory for station in network]
        
        # 2. HARVESTING
        for net, sta in available_stations:
            if sukses_unduh >= MAX_STATIONS_PER_EVENT:
                break 
                
            filename = f"{net}_{sta}_{eid}.mseed"
            file_path = os.path.join(event_dir, filename)
            
            # Skip jika file sudah sukses terunduh sebelumnya (Auto-Resume)
            if os.path.exists(file_path) and os.path.getsize(file_path) > 0:
                sukses_unduh += 1
                continue
                
            try:
                st = client.get_waveforms(
                    network=net, station=sta, location="*", channel="BH*,HH*,EH*",
                    starttime=t_event - 60, endtime=t_event + 240
                )
                
                # SENSOR QC (Validasi 3 Komponen)
                found_channels = set([tr.stats.channel[-1].upper() for tr in st])
                has_standard_nze = {'Z', 'N', 'E'}.issubset(found_channels)
                has_orthogonal_z12 = {'Z', '1', '2'}.issubset(found_channels)
                
                if not (has_standard_nze or has_orthogonal_z12):
                    del st
                    stasiun_gagal += 1
                    continue
                    
                # Simpan file biner
                os.makedirs(event_dir, exist_ok=True)
                st.write(file_path, format="MSEED")
                del st
                sukses_unduh += 1
                time.sleep(0.15) # Jeda sedikit lebih lama (150ms) untuk keamanan IP
                
            except Exception:
                stasiun_gagal += 1
                continue
                
        return eid, sukses_unduh, stasiun_gagal, "Selesai"
        
    except Exception as e:
        return eid, 0, 0, f"Error: {str(e)}"

# ==============================================================================
# 🚀 PIPELINE UTAMA
# ==============================================================================
def run_dynamic_pipeline_range():
    print(f"📡 Memulai Dynamic Harvesting untuk RENTANG TAHUN {START_YEAR} hingga {END_YEAR}...")
    df_master = pd.read_csv(PATH_KATALOG_MASTER)
    
    col_id = 'event_id'
    col_lat = 'latitude'
    col_lon = 'longitude'
    col_time = 'time_utc'
    col_mag = 'magnitude'
    
    df_master['waktu_dt'] = pd.to_datetime(df_master[col_time], errors='coerce', format='mixed')
    
    # 🔍 FILTER RENTANG TAHUN & MAGNITUDO
    df_target = df_master[
        (df_master['waktu_dt'].dt.year >= START_YEAR) & 
        (df_master['waktu_dt'].dt.year <= END_YEAR) & 
        (df_master[col_mag] >= MIN_MAGNITUDE)
    ].copy()
    
    if len(df_target) == 0:
        print(f"❌ Tidak ada data kejadian gempa untuk rentang {START_YEAR}-{END_YEAR} setelah filter.")
        return
        
    tasks = list(df_target[[col_id, col_time, col_lat, col_lon, col_mag]].itertuples(index=False, name=None))
    print(f"📊 Ditemukan {len(tasks)} kejadian gempa target (Mag >= {MIN_MAGNITUDE}).")
    print(f"🌍 Filter Network: {TARGET_NETWORKS} | Radius: {SEARCH_RADIUS_DEG}°")
    
    total_sukses = 0
    log_results = []
    
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(download_dynamic_event, task): task for task in tasks}
        
        with tqdm(total=len(tasks), desc="Processing Events", unit="event") as pbar:
            for future in as_completed(futures):
                eid, sukses, gagal, status = future.result()
                total_sukses += sukses
                log_results.append({'Event_ID': eid, 'File_Sukses': sukses, 'File_Gagal': gagal, 'Status': status})
                
                pbar.set_postfix({"Sukses Unduh": total_sukses})
                pbar.update(1)
                
                # Bebaskan RAM setiap 50 event
                if len(log_results) % 50 == 0:
                    gc.collect()

    pd.DataFrame(log_results).to_csv(LOG_DYNAMIC_SUCCESS, index=False)
    print("\n" + "="*50)
    print(f"🏁 DYNAMIC HARVESTING {START_YEAR}-{END_YEAR} SELESAI")
    print(f"✅ Total gelombang 3-komponen berhasil diunduh: {total_sukses} file")
    print("="*50)

if __name__ == "__main__":
    run_dynamic_pipeline_range()

📡 Memulai Dynamic Harvesting untuk RENTANG TAHUN 2006 hingga 2024...
📊 Ditemukan 176101 kejadian gempa target (Mag >= 2.5).
🌍 Filter Network: IA,GE,II | Radius: 5.0°


Processing Events: 100%|██████████| 176101/176101 [19:08:09<00:00,  2.56event/s, Sukses Unduh=52538]    



🏁 DYNAMIC HARVESTING 2006-2024 SELESAI
✅ Total gelombang 3-komponen berhasil diunduh: 52538 file


In [1]:
# -*- coding: utf-8 -*-
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
from obspy import read, UTCDateTime
from obspy.clients.fdsn import Client
from obspy.taup import TauPyModel
from obspy.geodetics import locations2degrees

# ==============================================================================
# 🎛️ KONFIGURASI PATH
# ==============================================================================
DIR_MSEED_RAW = r"/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/waveform_buffer/waveform_ready_02"
PATH_KATALOG = r"/Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/HYBRID_EARTHQUAKE_CATALOG_2001_2024_FIX.csv"

# Direktori Output 1-Komponen (1C) - Sumbu Vertikal (Z)
DIR_OUT_1C_GEMPA = r"/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/indonesia_waveform_master_data/output_data_npypure_1c_gempa_9s"
DIR_OUT_1C_NOISE = r"/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/indonesia_waveform_master_data/output_data_npypure_1c_noise_9s"

# Direktori Output 3-Komponen (3C) - Sumbu Timur, Utara, Vertikal (E, N, Z)
DIR_OUT_3C_GEMPA = r"/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/indonesia_waveform_master_data/output_data_npypure_3c_gempa_9s"
DIR_OUT_3C_NOISE = r"/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/indonesia_waveform_master_data/output_data_npypure_3c_noise_9s"

# Buat semua direktori
for d in [DIR_OUT_1C_GEMPA, DIR_OUT_1C_NOISE, DIR_OUT_3C_GEMPA, DIR_OUT_3C_NOISE]:
    os.makedirs(d, exist_ok=True)

# Inisialisasi Model
taup_model = TauPyModel(model="iasp91")
client = Client("EARTHSCOPE", timeout=30)
station_cache = {} 

def get_station_coords(net, sta):
    key = f"{net}_{sta}"
    if key in station_cache: return station_cache[key]
    try:
        inv = client.get_stations(network=net, station=sta, level="station")
        lat = inv[0][0].latitude
        lon = inv[0][0].longitude
        station_cache[key] = (lat, lon)
        return lat, lon
    except Exception:
        return None, None

# ==============================================================================
# 🚀 PROGRAM UTAMA (DUAL-SLICER 1C & 3C)
# ==============================================================================
def run_pure_dual_slicer():
    print("📡 Memuat Katalog Gempa Utama...")
    df_master = pd.read_csv(PATH_KATALOG, low_memory=False)
    df_master['event_id'] = df_master['event_id'].astype(str)
    catalog_dict = df_master.set_index('event_id').to_dict('index')

    print(f"🔍 Memindai file .mseed di direktori...")
    mseed_files = glob.glob(os.path.join(DIR_MSEED_RAW, "**", "*.mseed"), recursive=True)
    print(f"[INFO] Ditemukan {len(mseed_files):,} file mentah siap dipotong.\n")

    sukses_gempa = 0
    sukses_noise = 0
    
    with tqdm(total=len(mseed_files), desc="Memotong & Normalisasi (1C & 3C)") as pbar:
        for file_path in mseed_files:
            try:
                filename = os.path.basename(file_path)
                parts = filename.replace('.mseed', '').split('_')
                net, sta, eid = parts[0], parts[1], parts[2]

                if eid not in catalog_dict:
                    pbar.update(1)
                    continue

                event_info = catalog_dict[eid]
                eq_lat, eq_lon = float(event_info['latitude']), float(event_info['longitude'])
                eq_depth = float(event_info.get('depth', 10.0)) 
                origin_time = UTCDateTime(event_info['time_utc'])

                # 1. BACA & JAHIT GELOMBANG 
                st = read(file_path)
                st.merge(method=1, fill_value='interpolate') 
                st.interpolate(sampling_rate=100.0) # Resampling wajib 100Hz
                sr = 100.0
                
                # 2. EKSTRAKSI KOMPONEN KETAT (E, N, Z) atau (2, 1, Z)
                try:
                    st_Z = st.select(component="Z")[0]
                    # Mendukung arsitektur penamaan ortogonal (N/E atau 1/2)
                    st_N = st.select(component="N") if len(st.select(component="N")) > 0 else st.select(component="1")
                    st_E = st.select(component="E") if len(st.select(component="E")) > 0 else st.select(component="2")
                    st_N = st_N[0]
                    st_E = st_E[0]
                except IndexError:
                    pbar.update(1)
                    continue # Abaikan file jika tidak memiliki 3 komponen utuh
                
                # Sinkronisasi panjang array (menghindari error array dimensi)
                min_len = min(len(st_E.data), len(st_N.data), len(st_Z.data))
                data_E = st_E.data[:min_len]
                data_N = st_N.data[:min_len]
                data_Z = st_Z.data[:min_len]

                # Format Matriks 3C: Kolom 0=Timur(E), Kolom 1=Utara(N), Kolom 2=Vertikal(Z)
                data_3c_master = np.column_stack((data_E, data_N, data_Z)).astype(np.float32)
                # Format Matriks 1C: Hanya Vertikal(Z)
                data_1c_master = data_Z.astype(np.float32)

                # ==============================================================
                # [EKSTRAKSI 1]: PURE AMBIENT NOISE (DERAU)
                # ==============================================================
                noise_start_idx = int(10 * sr) # Ambil detik ke-10 (jauh sebelum gempa)
                noise_9s_idx = noise_start_idx + 900
                noise_7s_idx = noise_start_idx + 700
                
                if noise_9s_idx < min_len:
                    # PROSES 1C NOISE
                    noise_9s_1c = data_1c_master[noise_start_idx:noise_9s_idx].copy()
                    noise_9s_1c -= np.mean(noise_9s_1c)
                    norm_1c = np.max(np.abs(noise_9s_1c))
                    
                    noise_7s_1c = data_1c_master[noise_start_idx:noise_7s_idx].copy()
                    noise_7s_1c -= np.mean(noise_7s_1c)
                    if norm_1c > 0:
                        noise_7s_1c /= norm_1c
                        np.save(os.path.join(DIR_OUT_1C_NOISE, f"NOISE_1C_{filename}.npy"), noise_7s_1c)

                    # PROSES 3C NOISE
                    noise_9s_3c = data_3c_master[noise_start_idx:noise_9s_idx].copy()
                    for c in range(3): noise_9s_3c[:, c] -= np.mean(noise_9s_3c[:, c])
                    norm_3c = np.max(np.abs(noise_9s_3c)) # Global Max 3C
                    
                    noise_7s_3c = data_3c_master[noise_start_idx:noise_7s_idx].copy()
                    for c in range(3): noise_7s_3c[:, c] -= np.mean(noise_7s_3c[:, c])
                    if norm_3c > 0:
                        noise_7s_3c /= norm_3c
                        np.save(os.path.join(DIR_OUT_3C_NOISE, f"NOISE_3C_{filename}.npy"), noise_7s_3c)
                    
                    sukses_noise += 1

                # ==============================================================
                # [EKSTRAKSI 2]: PURE EARTHQUAKE (GEMPA P-ARRIVAL)
                # ==============================================================
                sta_lat, sta_lon = get_station_coords(net, sta)
                if sta_lat is None:
                    pbar.update(1)
                    continue
                    
                dist_deg = locations2degrees(eq_lat, eq_lon, sta_lat, sta_lon)
                arrivals = taup_model.get_travel_times(source_depth_in_km=eq_depth, distance_in_degree=dist_deg, phase_list=["P", "Pn", "Pg"])
                if len(arrivals) == 0:
                    pbar.update(1)
                    continue
                
                p_arrival_time = origin_time + arrivals[0].time
                p_idx = int((p_arrival_time - st_Z.stats.starttime) * sr)
                eq_9s_idx = p_idx + 900
                eq_7s_idx = p_idx + 700
                
                if p_idx >= 0 and eq_9s_idx < min_len:
                    # PROSES 1C GEMPA
                    eq_9s_1c = data_1c_master[p_idx:eq_9s_idx].copy()
                    eq_9s_1c -= np.mean(eq_9s_1c)
                    norm_eq_1c = np.max(np.abs(eq_9s_1c))
                    
                    eq_7s_1c = data_1c_master[p_idx:eq_7s_idx].copy()
                    eq_7s_1c -= np.mean(eq_7s_1c)
                    if norm_eq_1c > 0:
                        eq_7s_1c /= norm_eq_1c
                        np.save(os.path.join(DIR_OUT_1C_GEMPA, f"GEMPA_1C_{filename}.npy"), eq_7s_1c)

                    # PROSES 3C GEMPA
                    eq_9s_3c = data_3c_master[p_idx:eq_9s_idx].copy()
                    for c in range(3): eq_9s_3c[:, c] -= np.mean(eq_9s_3c[:, c])
                    norm_eq_3c = np.max(np.abs(eq_9s_3c)) # Global Max 3C
                    
                    eq_7s_3c = data_3c_master[p_idx:eq_7s_idx].copy()
                    for c in range(3): eq_7s_3c[:, c] -= np.mean(eq_7s_3c[:, c])
                    if norm_eq_3c > 0:
                        eq_7s_3c /= norm_eq_3c
                        np.save(os.path.join(DIR_OUT_3C_GEMPA, f"GEMPA_3C_{filename}.npy"), eq_7s_3c)
                        
                    sukses_gempa += 1

            except Exception as e:
                pass 
            finally:
                pbar.update(1)

    print("\n" + "="*50)
    print("🏁 EKSTRAKSI DUAL-SLICER (1C & 3C) SELESAI!")
    print(f"✅ Berhasil memproduksi: {sukses_noise:,} pasang file NOISE (1C & 3C)")
    print(f"✅ Berhasil memproduksi: {sukses_gempa:,} pasang file GEMPA (1C & 3C)")
    print("="*50)

if __name__ == "__main__":
    run_pure_dual_slicer()

📡 Memuat Katalog Gempa Utama...
🔍 Memindai file .mseed di direktori...
[INFO] Ditemukan 52,761 file mentah siap dipotong.



Memotong & Normalisasi (1C & 3C): 52993it [19:33, 45.07it/s]                            /opt/homebrew/Caskroom/miniforge/base/envs/mcu_quake_env/lib/python3.10/site-packages/obspy/signal/interpolation.py:142: RuntimeWarning: divide by zero encountered in divide
  w = 1.0 / np.clip(w, np.spacing(1), w.max())
/opt/homebrew/Caskroom/miniforge/base/envs/mcu_quake_env/lib/python3.10/site-packages/obspy/signal/interpolation.py:146: RuntimeWarning: invalid value encountered in multiply
  slope[1:-1] = (w[:-1] * m[:-1] + w[1:] * m[1:]) / (w[:-1] + w[1:])
Memotong & Normalisasi (1C & 3C): 53082it [19:35, 45.17it/s]



🏁 EKSTRAKSI DUAL-SLICER (1C & 3C) SELESAI!
✅ Berhasil memproduksi: 52,758 pasang file NOISE (1C & 3C)
✅ Berhasil memproduksi: 52,427 pasang file GEMPA (1C & 3C)


In [2]:
# -*- coding: utf-8 -*-
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm
from obspy import read, UTCDateTime
from obspy.clients.fdsn import Client
from obspy.taup import TauPyModel
from obspy.geodetics import locations2degrees

# ==============================================================================
# 🎛️ KONFIGURASI PATH (UPDATED MASTER DATA)
# ==============================================================================
DIR_MSEED_RAW = r"/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/output/waveform_buffer/waveform_ready_02"
PATH_KATALOG = r"/Volumes/Local Disk/Code_Git/S3_code/seismic/Indonesian_Earthquake_Catalog_BMKG_1998_2024/HYBRID_EARTHQUAKE_CATALOG_2001_2024_FIX.csv"

# Direktori Output 1-Komponen (1C) - Sumbu Vertikal (Z)
DIR_OUT_1C_GEMPA = r"/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/indonesia_waveform_master_data/output_data_npypure_1c_gempa_9s"
DIR_OUT_1C_NOISE = r"/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/indonesia_waveform_master_data/output_data_npypure_1c_noise_9s"

# Direktori Output 3-Komponen (3C) - Sumbu Timur, Utara, Vertikal (E, N, Z)
DIR_OUT_3C_GEMPA = r"/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/indonesia_waveform_master_data/output_data_npypure_3c_gempa_9s"
DIR_OUT_3C_NOISE = r"/Volumes/Local Disk/Code_Git/S3_code/seismic/mcu_quake/code_gen_trying/indonesia_jaya_benchmarking_data/indonesia_waveform_master_data/output_data_npypure_3c_noise_9s"

for d in [DIR_OUT_1C_GEMPA, DIR_OUT_1C_NOISE, DIR_OUT_3C_GEMPA, DIR_OUT_3C_NOISE]:
    os.makedirs(d, exist_ok=True)

# Inisialisasi Model TauP
taup_model = TauPyModel(model="iasp91")
client = Client("EARTHSCOPE", timeout=30)
station_cache = {} 

def get_station_coords(net, sta):
    key = f"{net}_{sta}"
    if key in station_cache: return station_cache[key]
    try:
        inv = client.get_stations(network=net, station=sta, level="station")
        lat = inv[0][0].latitude
        lon = inv[0][0].longitude
        station_cache[key] = (lat, lon)
        return lat, lon
    except Exception:
        return None, None

# ==============================================================================
# 🚀 PROGRAM UTAMA
# ==============================================================================
def run_pure_dual_slicer_corrected():
    print("📡 Memuat Katalog Gempa Utama...")
    df_master = pd.read_csv(PATH_KATALOG, low_memory=False)
    df_master['event_id'] = df_master['event_id'].astype(str)
    catalog_dict = df_master.set_index('event_id').to_dict('index')

    mseed_files = glob.glob(os.path.join(DIR_MSEED_RAW, "**", "*.mseed"), recursive=True)
    print(f"[INFO] Ditemukan {len(mseed_files):,} file mentah siap dipotong.\n")

    sukses_ekstraksi = 0
    
    with tqdm(total=len(mseed_files), desc="Slicing & Preserving SNR") as pbar:
        for file_path in mseed_files:
            try:
                filename = os.path.basename(file_path)
                parts = filename.replace('.mseed', '').split('_')
                net, sta, eid = parts[0], parts[1], parts[2]

                if eid not in catalog_dict:
                    pbar.update(1)
                    continue

                event_info = catalog_dict[eid]
                eq_lat, eq_lon = float(event_info['latitude']), float(event_info['longitude'])
                eq_depth = float(event_info.get('depth', 10.0)) 
                origin_time = UTCDateTime(event_info['time_utc'])

                # 1. BACA, JAHIT, RESAMPLING
                st = read(file_path)
                st.merge(method=1, fill_value='interpolate') 
                st.interpolate(sampling_rate=100.0) 
                sr = 100.0
                
                # 2. EKSTRAKSI KOMPONEN KETAT
                try:
                    st_Z = st.select(component="Z")[0]
                    st_N = st.select(component="N") if len(st.select(component="N")) > 0 else st.select(component="1")
                    st_E = st.select(component="E") if len(st.select(component="E")) > 0 else st.select(component="2")
                    st_N = st_N[0]
                    st_E = st_E[0]
                except IndexError:
                    pbar.update(1)
                    continue 
                
                min_len = min(len(st_E.data), len(st_N.data), len(st_Z.data))
                data_3c_master = np.column_stack((st_E.data[:min_len], st_N.data[:min_len], st_Z.data[:min_len])).astype(np.float32)
                data_1c_master = st_Z.data[:min_len].astype(np.float32)

                # ==============================================================
                # LANGKAH 1: CARI P-ARRIVAL & HITUNG NILAI PEMBAGI (NORM_EQ)
                # ==============================================================
                sta_lat, sta_lon = get_station_coords(net, sta)
                if sta_lat is None:
                    pbar.update(1)
                    continue
                    
                dist_deg = locations2degrees(eq_lat, eq_lon, sta_lat, sta_lon)
                arrivals = taup_model.get_travel_times(source_depth_in_km=eq_depth, distance_in_degree=dist_deg, phase_list=["P", "Pn", "Pg"])
                if len(arrivals) == 0:
                    pbar.update(1)
                    continue
                
                p_arrival_time = origin_time + arrivals[0].time
                p_idx = int((p_arrival_time - st_Z.stats.starttime) * sr)
                
                eq_9s_idx = p_idx + 900
                eq_7s_idx = p_idx + 700
                
                # Pastikan P-arrival valid dan tidak keluar batas
                if p_idx < 0 or eq_9s_idx >= min_len:
                    pbar.update(1)
                    continue

                # Kunci Normalisasi 1C (Max Absolut 9 detik setelah P-arrival)
                eq_9s_1c = data_1c_master[p_idx:eq_9s_idx].copy()
                eq_9s_1c -= np.mean(eq_9s_1c)
                norm_eq_1c = np.max(np.abs(eq_9s_1c))
                
                # Kunci Normalisasi 3C (Global Max Absolut 9 detik setelah P-arrival)
                eq_9s_3c = data_3c_master[p_idx:eq_9s_idx].copy()
                for c in range(3): eq_9s_3c[:, c] -= np.mean(eq_9s_3c[:, c])
                norm_eq_3c = np.max(np.abs(eq_9s_3c))

                # Jika pembagi 0 (data flat/rusak), lewati file ini
                if norm_eq_1c <= 0 or norm_eq_3c <= 0:
                    pbar.update(1)
                    continue

                # ==============================================================
                # LANGKAH 2: EKSTRAKSI & SIMPAN GEMPA (Menggunakan NORM_EQ)
                # ==============================================================
                eq_7s_1c = data_1c_master[p_idx:eq_7s_idx].copy()
                eq_7s_1c -= np.mean(eq_7s_1c)
                eq_7s_1c /= norm_eq_1c  # <-- Normalisasi Gempa 1C
                np.save(os.path.join(DIR_OUT_1C_GEMPA, f"GEMPA_1C_{filename}.npy"), eq_7s_1c)

                eq_7s_3c = data_3c_master[p_idx:eq_7s_idx].copy()
                for c in range(3): 
                    eq_7s_3c[:, c] -= np.mean(eq_7s_3c[:, c])
                eq_7s_3c /= norm_eq_3c  # <-- Normalisasi Gempa 3C
                np.save(os.path.join(DIR_OUT_3C_GEMPA, f"GEMPA_3C_{filename}.npy"), eq_7s_3c)

                # ==============================================================
                # LANGKAH 3: EKSTRAKSI & SIMPAN NOISE (Menggunakan NORM_EQ yang SAMA!)
                # ==============================================================
                # Ambil 7 detik Noise jauh sebelum P-Arrival (misal dari detik ke-10)
                noise_start_idx = int(10 * sr) 
                noise_7s_idx = noise_start_idx + 700
                
                # Pastikan noise tidak menabrak gelombang P
                if noise_7s_idx < p_idx:
                    noise_7s_1c = data_1c_master[noise_start_idx:noise_7s_idx].copy()
                    noise_7s_1c -= np.mean(noise_7s_1c)
                    noise_7s_1c /= norm_eq_1c  # <-- SNR TERJAGA! Dibagi dengan max Gempa
                    np.save(os.path.join(DIR_OUT_1C_NOISE, f"NOISE_1C_{filename}.npy"), noise_7s_1c)

                    noise_7s_3c = data_3c_master[noise_start_idx:noise_7s_idx].copy()
                    for c in range(3): 
                        noise_7s_3c[:, c] -= np.mean(noise_7s_3c[:, c])
                    noise_7s_3c /= norm_eq_3c  # <-- SNR TERJAGA! Dibagi dengan max Gempa
                    np.save(os.path.join(DIR_OUT_3C_NOISE, f"NOISE_3C_{filename}.npy"), noise_7s_3c)

                sukses_ekstraksi += 1

            except Exception as e:
                pass 
            finally:
                pbar.update(1)

    print("\n" + "="*50)
    print("🏁 EKSTRAKSI DUAL-SLICER (KOREKSI SNR & P-ARRIVAL) SELESAI!")
    print(f"✅ Berhasil memproduksi: {sukses_ekstraksi:,} set data seimbang (Gempa & Noise | 1C & 3C)")
    print("="*50)

if __name__ == "__main__":
    run_pure_dual_slicer_corrected()

📡 Memuat Katalog Gempa Utama...
[INFO] Ditemukan 52,761 file mentah siap dipotong.



Slicing & Preserving SNR: 53006it [20:05, 41.76it/s]                            /opt/homebrew/Caskroom/miniforge/base/envs/mcu_quake_env/lib/python3.10/site-packages/obspy/signal/interpolation.py:142: RuntimeWarning: divide by zero encountered in divide
  w = 1.0 / np.clip(w, np.spacing(1), w.max())
/opt/homebrew/Caskroom/miniforge/base/envs/mcu_quake_env/lib/python3.10/site-packages/obspy/signal/interpolation.py:146: RuntimeWarning: invalid value encountered in multiply
  slope[1:-1] = (w[:-1] * m[:-1] + w[1:] * m[1:]) / (w[:-1] + w[1:])
Slicing & Preserving SNR: 53095it [20:07, 43.98it/s]



🏁 EKSTRAKSI DUAL-SLICER (KOREKSI SNR & P-ARRIVAL) SELESAI!
✅ Berhasil memproduksi: 52,427 set data seimbang (Gempa & Noise | 1C & 3C)
